# Debate / Adversarial Collaboration | Multi-Agent Collaboration

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import TypedDict, List, Literal
from typing_extensions import NotRequired
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
class DebateState(TypedDict):
    topic: str
    pro_arguments: List[str]
    con_arguments: List[str]
    round: NotRequired[int]
    verdict: NotRequired[str]

MAX_ROUNDS = 2

In [5]:
def proponent(state: DebateState) -> dict:
    """Argue FOR the proposition."""
    previous_con = state["con_arguments"][-1] if state.get("con_arguments") else "No counter-arguments yet."
    previous_pro = state["pro_arguments"][-1] if state.get("pro_arguments") else ""
    response = model.invoke(
        f"You are arguing FOR this proposition: '{state['topic']}'\n\n"
        f"Your previous argument: {previous_pro or 'This is your opening statement.'}\n"
        f"Opponent's latest counter: {previous_con}\n\n"
        f"Provide a strong, evidence-based argument. Address the opponent's points directly."
    )
    new_args = list(state.get("pro_arguments", [])) + [response.content]
    return {"pro_arguments": new_args}

def opponent(state: DebateState) -> Command[Literal["proponent", "judge"]]:
    """Argue AGAINST the proposition."""
    previous_pro = state["pro_arguments"][-1] if state.get("pro_arguments") else "No arguments yet."
    previous_con = state["con_arguments"][-1] if state.get("con_arguments") else ""
    response = model.invoke(
        f"You are arguing AGAINST this proposition: '{state['topic']}'\n\n"
        f"Your previous argument: {previous_con or 'This is your opening statement.'}\n"
        f"Proponent's latest argument: {previous_pro}\n\n"
        f"Provide a strong counter-argument. Identify weaknesses and present alternative evidence."
    )
    new_args = list(state.get("con_arguments", [])) + [response.content]
    new_round = state.get("round", 0) + 1
    update = {"con_arguments": new_args, "round": new_round}
    if new_round >= MAX_ROUNDS:
        return Command(goto="judge", update=update)
    return Command(goto="proponent", update=update)

def judge(state: DebateState) -> dict:
    """Evaluate the debate and render a verdict."""
    debate_log = ""
    for i in range(len(state.get("pro_arguments", []))):
        debate_log += f"\n--- Round {i+1} ---\n"
        debate_log += f"PRO: {state['pro_arguments'][i]}\n"
        if i < len(state.get("con_arguments", [])):
            debate_log += f"CON: {state['con_arguments'][i]}\n"
    response = model.invoke(
        f"You are an impartial judge evaluating a debate.\n\n"
        f"Proposition: '{state['topic']}'\n\n"
        f"Debate Transcript:\n{debate_log}\n\n"
        f"Provide:\n"
        f"1. Strongest arguments from each side\n"
        f"2. Weaknesses in each side's reasoning\n"
        f"3. Your verdict (support, oppose, or nuanced position)\n"
        f"4. Confidence level and key factors in your decision"
    )
    return {"verdict": response.content}

In [6]:
graph = StateGraph(DebateState)
graph.add_node("proponent", proponent)
graph.add_node("opponent", opponent, destinations=("proponent", "judge"))
graph.add_node("judge", judge)

graph.add_edge(START, "proponent")
graph.add_edge("proponent", "opponent")
# opponent returns Command to route directly
graph.add_edge("judge", END)

debate = graph.compile()

In [7]:
# Plot the workflow
plot_mermaid(debate)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	proponent(proponent)
	opponent(opponent)
	judge(judge)
	__end__([<p>__end__</p>]):::last
	__start__ --> proponent;
	opponent -.-> judge;
	opponent -.-> proponent;
	proponent --> opponent;
	judge --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [8]:
result = debate.invoke({
    "topic": "Companies should mandate return-to-office 5 days a week for software engineers",
    "pro_arguments": [],
    "con_arguments": [],
})
print(result["verdict"])

### Evaluation of the Debate

#### 1. Strongest Arguments

**PRO:**  
- **Collaboration and Innovation:** The argument that in-person work fosters spontaneous interactions that facilitate innovation is compelling. The reference to studies demonstrating how physical proximity enhances creative output and problem-solving supports the claim that in-office work can drive essential innovation in the tech industry, which thrives on breakthroughs and spontaneous collaboration.

**CON:**  
- **Remote Work Benefits & Global Talent:** The strongest argument from the CON side emphasizes the benefits of remote work in terms of accessing global talent and offering flexible work-life balance. This argument is bolstered by statistics indicating increased job satisfaction and improved balance, which can enhance competitive advantages for companies that adapt to modern, flexible work arrangements.

#### 2. Weaknesses in Each Side's Reasoning

**PRO:**
- **Over-reliance on Traditional Dynamics:** The PR

In [9]:
stream_invoke(debate, {
    "topic": "Companies should mandate return-to-office 5 days a week for software engineers",
    "pro_arguments": [],
    "con_arguments": [],
})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'topic': 'Companies should mandate return-to-office 5 days a week for software engineers',
 'pro_arguments': ['While the flexibility of remote work has been a crucial advantage in recent times, there are compelling reasons for companies to mandate a return-to-office policy for software engineers five days a week. This argument rests on three central pillars: collaboration and innovation, corporate culture and employee development, and data security and infrastructure.\n\n1. **Collaboration and Innovation**: In-person work significantly enhances the capacity for spontaneous collaboration and creativity. A study from MIT found that physical proximity between team members leads to higher frequencies of informal interactions, which are crucial for idea generation and problem-solving. In-office work fosters an environment where brainstorming sessions can happen organically, and immediate feedback can be exchanged rapidly, which is much harder to achieve in a virtual setting. The tech indus